# VS_Chance_AlarmEval — Chance Baselines with Alarm Aggregation

**Purpose.** Recompute the three chance baselines from
`VS_Chance_Baseline.ipynb` (majority-class, stratified-random, uniform-random)
applying the literature-standard K=5/M=12 sliding-window vote with 30-min
refractory. Trivial compute — no model training. Uses the V3 GC cache only
for window counts (the labels are what matter; features don't enter).


In [1]:
# --- portable repo bootstrap (added for public release; locates the repo root) ---
import sys as _sys, pathlib as _pl
REPO = _pl.Path.cwd()
while not (REPO / 'src' / 'config.py').exists() and REPO != REPO.parent:
    REPO = REPO.parent
_sys.path.insert(0, str(REPO / 'src'))
from pathlib import Path
CODE_DIR = str(REPO); CODE = REPO; CODEV2 = REPO; PROJECT_DIR = REPO
# --------------------------------------------------------------------------------

# Cell 0 — Imports
import os, sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

# [path set by bootstrap] CODE_DIR = r"<repo>/Code"
sys.path.insert(0, CODE_DIR)

from config import (
    DATA_ROOT, STEP_SEC, EXCLUDED_PATIENTS,
    GC_MATRICES_DIR_V3, RESULTS_DIR, RANDOM_SEED,
    INTERICTAL_MULTIPLIER, MAX_INTERICTAL_ABS,
    ALARM_K, ALARM_M, ALARM_REFRACTORY,
)
from metrics import evaluate_with_alarms

np.random.seed(RANDOM_SEED)
print(f'Alarm K={ALARM_K}/M={ALARM_M}, refractory={ALARM_REFRACTORY*STEP_SEC//60}min')


Alarm K=5/M=12, refractory=30min


In [2]:
# Cell 1 — Load labels per patient (we only need y, not the GC matrices)
cache_root = Path(GC_MATRICES_DIR_V3)
assert cache_root.exists()

patients_all = sorted([
    p for p in os.listdir(DATA_ROOT)
    if os.path.isdir(os.path.join(DATA_ROOT, p))
    and p.startswith('chb') and p not in EXCLUDED_PATIENTS
])

patient_labels = {}
for pid in patients_all:
    pdir = cache_root / pid
    if not pdir.exists(): continue
    lab_files = sorted(pdir.glob('*_labels.npy'))
    if not lab_files: continue
    y = np.concatenate([np.load(f) for f in lab_files]).astype(np.int8)
    n_pre = int((y==1).sum()); n_int = int((y==0).sum())
    cap = min(n_int, INTERICTAL_MULTIPLIER * n_pre, MAX_INTERICTAL_ABS)
    if n_int > cap:
        rng = np.random.default_rng(RANDOM_SEED + hash(pid) % 10_000)
        int_idx = np.where(y == 0)[0]; pre_idx = np.where(y == 1)[0]
        keep = np.sort(np.concatenate([pre_idx, rng.choice(int_idx, size=cap, replace=False)]))
        y = y[keep]
    if n_pre == 0: continue
    patient_labels[pid] = y

patient_ids = sorted(patient_labels.keys())
print(f'Loaded labels for {len(patient_ids)} patients')


Loaded labels for 21 patients


In [3]:
# Cell 2 — Generate chance predictions and evaluate at both window and alarm level

def chance_probs(y, mode, rng):
    n = len(y)
    if mode == 'majority':
        # Predict majority class (interictal in CHB-MIT)
        return np.zeros(n)
    elif mode == 'uniform':
        return rng.uniform(0, 1, n)
    elif mode == 'stratified':
        # Predict positive at the empirical preictal prevalence
        prev = (y == 1).mean()
        return rng.uniform(0, 1, n) < prev  # bool, will threshold at 0.5
    else:
        raise ValueError(mode)

MODES = ['majority', 'uniform', 'stratified']

results_alarm, results_cmp = {m: [] for m in MODES}, {m: [] for m in MODES}
for pid in patient_ids:
    y = patient_labels[pid]
    rng = np.random.default_rng(RANDOM_SEED + hash(pid) % 10_000)
    for mode in MODES:
        probs = chance_probs(y, mode, rng).astype(float)
        if probs.std() < 1e-12:
            # AUC undefined for constant predictor → skip alarm eval
            continue
        threshold = 0.5
        try:
            a = evaluate_with_alarms(
                y_true=y, probs=probs, threshold=threshold,
                K=ALARM_K, M=ALARM_M, refractory_windows=ALARM_REFRACTORY,
                patient_id=pid,
            )
        except Exception:
            continue
        a['mode'] = mode
        results_alarm[mode].append(a)

        pred = (probs >= threshold).astype(int)
        tp=int(((pred==1)&(y==1)).sum()); fp=int(((pred==1)&(y==0)).sum())
        tn=int(((pred==0)&(y==0)).sum()); fn=int(((pred==0)&(y==1)).sum())
        sens_w = tp/max(tp+fn,1)
        hours_int = (y==0).sum() * STEP_SEC / 3600.0
        fpr_w = fp/max(hours_int, 1e-9)
        results_cmp[mode].append(dict(patient=pid, mode=mode,
            auc=a['auc'], auc_pr=a['auc_pr'],
            sens_window=sens_w, fpr_h_window=fpr_w,
            sens_alarm=a['sensitivity'], fpr_h_alarm=a['fpr_per_hour']))

print('Chance baselines evaluated.')


Chance baselines evaluated.


In [4]:
# Cell 3 — Save and summarise
print(f'{"Mode":<12} {"AUC":>6} {"AUC-PR":>7} '
      f'{"Sens(w)":>8} {"FPR/h(w)":>10} {"Sens(a)":>8} {"FPR/h(a)":>10}')
print('-' * 70)
for mode in MODES:
    if not results_alarm[mode]:
        print(f'{mode:<12} (skipped — constant predictor)')
        continue
    df_a = pd.DataFrame(results_alarm[mode])
    df_c = pd.DataFrame(results_cmp[mode])
    df_a.to_csv(Path(RESULTS_DIR) / f'lopo_chance_{mode}_alarm.csv', index=False)
    df_c.to_csv(Path(RESULTS_DIR) / f'lopo_chance_{mode}_compare.csv', index=False)
    print(f'{mode:<12} {df_c.auc.mean():>6.3f} {df_c.auc_pr.mean():>7.3f} '
          f'{df_c.sens_window.mean():>8.3f} {df_c.fpr_h_window.mean():>10.1f} '
          f'{df_c.sens_alarm.mean():>8.3f} {df_c.fpr_h_alarm.mean():>10.2f}')

print('\nOutputs: lopo_chance_{majority,uniform,stratified}_{alarm,compare}.csv')


Mode            AUC  AUC-PR  Sens(w)   FPR/h(w)  Sens(a)   FPR/h(a)
----------------------------------------------------------------------
majority     (skipped — constant predictor)
uniform       0.504   0.172    0.501      178.9    0.005       2.04
stratified    0.500   0.168    0.164       59.2    0.003       1.23

Outputs: lopo_chance_{majority,uniform,stratified}_{alarm,compare}.csv
